In [ ]:
import pandas as pd
from pathlib import Path

carpeta = Path(".")

df = pd.read_csv("master.csv")

dfs = {}

for archivo in carpeta.glob("*.csv"):
    dfs[archivo.name] = pd.read_csv(archivo)


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,...,pct_transacciones_no_procesadas,n_conversaciones,total_mensajes,total_negativos,total_positivos,total_neutrales,score_sentimiento_promedio,pct_mensajes_negativos,pct_mensajes_positivos,satisfaccion_cliente
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,...,0.053571,1,3,0,0,3,0.844355,0.0,0.000000,Neutral
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,...,0.040541,1,3,0,1,2,0.754270,0.0,0.333333,Alta satisfacción
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,...,0.101266,1,3,0,0,3,0.824913,0.0,0.000000,Neutral
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,...,0.075758,1,3,0,0,3,0.850217,0.0,0.000000,Neutral
4,USR-00005,26,M,Ciudad de México,CDMX - Cuauhtémoc,Preparatoria,Empresario,27000,816,True,...,0.043011,1,5,0,0,5,0.762400,0.0,0.000000,Neutral


In [5]:
master = dfs["master.csv"]

master["satisfaccion_cliente"].value_counts()

import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
pip install umap-learn plotly

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 21.5 MB/s  0:00:00
   ---------------------------------------- 0.0/39.2 MB ? eta -:--:--
   ------ --------------------------------- 6.6/39.2 MB 31.3 MB/s eta 0:00:02
   ------------- -------------------------- 13.1/39.2 MB 30.7 MB/s eta 0:00:01
   ------------------ --------------------- 18.4/39.2 MB 29.3 MB/s eta 0:00:01
   ------------------------- -------------- 24.6/39.2 MB 29.4 MB/s eta 0:00:01
   -------------------------------- ------- 31.7/39.2 MB 30.3 MB/s eta 0:00:01
   ---------------------------------------  39.1/39.2 MB 31.0 MB/s eta 0:00:01
   ---------------------------------------  39.1/39.2 MB 31.0 MB/s eta 0:00:01
   ---------------------------------------- 39.2/39.2 MB 24.9 MB/s  0:00:01

   ---------------------------------------- 0/4 [llvmlite]
   ---------------------------------------- 0/4 [llvmlite]
   -------------------------------------

# HDBSCAN

In [8]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap
import hdbscan

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import plotly.express as px

# ====================================================
# 1. Cargar base
# ====================================================

df = pd.read_csv("master.csv")

# Guardar satisfacción
y_sat = df["satisfaccion_cliente"]

# ====================================================
# 2. Variables numéricas
# ====================================================

X = df.drop(columns=["user_id", "satisfaccion_cliente"])

X = X.select_dtypes(include=["int64","float64","bool"])

for c in X.select_dtypes(include=["bool"]).columns:
    X[c] = X[c].astype(int)

X = X.fillna(0)

# ====================================================
# 3. Escalar
# ====================================================

X_scaled = StandardScaler().fit_transform(X)

# ====================================================
# 4. UMAP
# ====================================================

embed = umap.UMAP(
    n_neighbors=20,
    min_dist=0.05,
    n_components=3,
    random_state=42
).fit_transform(X_scaled)

# ====================================================
# 5. Detectar islas
# ====================================================

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=30
)

clusters = clusterer.fit_predict(embed)

df["cluster"] = clusters

# ====================================================
# 6. Visualización
# ====================================================

plot_df = pd.DataFrame(embed, columns=["UMAP1","UMAP2","UMAP3"])
plot_df["cluster"] = clusters.astype(str)

fig = px.scatter_3d(
    plot_df,
    x="UMAP1", y="UMAP2", z="UMAP3",
    color="cluster",
    title="Islas detectadas por HDBSCAN"
)

fig.show()

# ====================================================
# 7. Qué variables separan clusters
# ====================================================

df_model = X.copy()
df_model["cluster"] = clusters

df_model = df_model[df_model["cluster"] != -1]

Xrf = df_model.drop(columns="cluster")
yrf = df_model["cluster"]

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(Xrf, yrf)

imp = pd.DataFrame({
    "Variable": Xrf.columns,
    "Importancia": rf.feature_importances_
}).sort_values("Importancia", ascending=False)

print(imp.head(15))

# ====================================================
# 8. Cruce con satisfacción
# ====================================================

tabla = pd.crosstab(
    df["cluster"],
    df["satisfaccion_cliente"],
    normalize="index"
) * 100

print(tabla.round(2))

ModuleNotFoundError: No module named 'hdbscan'

$$
P(\text{abandono}_i)=\frac{1}{1+e^{-Z_i}}
$$

$$
Z_i=\beta_0+\sum_{j=1}^{p}\alpha_j\,z(x_{ij})+\lambda\,w_{c_i}+\gamma_{\text{sat}(i)}
$$

$$
\alpha_j=s_j\frac{I_j}{\sum_{k=1}^{p}I_k}
$$

$$
w_{c_i}=\log\left(\frac{r_{c_i}+\epsilon}{1-r_{c_i}+\epsilon}\right)
$$

$$
r_{c_i}=P(\text{Baja satisfacción}\mid \text{cluster}=c_i)
$$

$$
\gamma_{\text{sat}(i)}=
\begin{cases}
-\eta, & \text{Alta satisfacción} \\
0, & \text{Neutral} \\
+\eta, & \text{Baja satisfacción}
\end{cases}
$$

$$
s_j=
\begin{cases}
+1, & \text{si la variable aumenta el riesgo} \\
-1, & \text{si la variable reduce el riesgo}
\end{cases}
$$

$$
\boxed{
P(\text{abandono}_i)=
\frac{1}{
1+\exp\left[
-\left(
\beta_0+
\sum_{j=1}^{p}
s_j\frac{I_j}{\sum_{k=1}^{p}I_k}z(x_{ij})
+\lambda w_{c_i}
+\gamma_{\text{sat}(i)}
\right)
\right]
}
}
$$

# Función predictiva

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. Importancias obtenidas del modelo
# ============================================================

importancias = {
    "monto_maximo": 0.084690,
    "recibe_remesas": 0.078560,
    "utilizacion_promedio": 0.064548,
    "n_transacciones": 0.056276,
    "monto_total": 0.054234,
    "cashback_total": 0.053843,
    "ticket_promedio": 0.050921,
    "dias_desde_ultimo_login": 0.050062,
    "usa_hey_shop": 0.044159,
    "pct_mensajes_positivos": 0.043005,
    "pct_patron_atipico_tx": 0.040301,
    "patron_uso_atipico": 0.038111,
    "nomina_domiciliada": 0.037680,
    "total_positivos": 0.036864,
    "n_productos_cancelados": 0.025953
}

# ============================================================
# 2. Dirección esperada del efecto sobre abandono
# ============================================================
# +1: aumenta el riesgo de abandono
# -1: reduce el riesgo de abandono

signos = {
    "monto_maximo": -1,
    "recibe_remesas": -1,
    "utilizacion_promedio": 1,
    "n_transacciones": -1,
    "monto_total": -1,
    "cashback_total": -1,
    "ticket_promedio": -1,
    "dias_desde_ultimo_login": 1,
    "usa_hey_shop": -1,
    "pct_mensajes_positivos": -1,
    "pct_patron_atipico_tx": 1,
    "patron_uso_atipico": 1,
    "nomina_domiciliada": -1,
    "total_positivos": -1,
    "n_productos_cancelados": 1
}

# ============================================================
# 3. Proporción de baja satisfacción por cluster
# ============================================================

baja_satisfaccion_cluster = {
    -1: 0.3362,
     0: 0.2927,
     1: 0.3263,
     2: 0.3255,
     3: 0.3201,
     4: 0.3518,
     5: 0.3281,
     6: 0.2986,
     7: 0.3504,
     8: 0.3745,
     9: 0.2804,
    10: 0.1918,
    11: 0.3358,
    12: 0.2701,
    13: 0.3043,
    14: 0.3652,
    15: 0.1760,
    16: 0.2802,
    17: 0.0000,
    18: 0.8369
}

In [ ]:
def predecir_abandono(
    df,
    importancias,
    signos,
    baja_satisfaccion_cluster,
    beta_0=-1.5,
    lambda_cluster=0.7,
    eta=1.2,
    epsilon=0.001
):
    """
    Calcula una probabilidad proxy de abandono para cada cliente.

    Parámetros
    ----------
    df : DataFrame
        Base de datos que debe contener las variables de importancia,
        la columna 'cluster' y la columna 'satisfaccion_cliente'.

    importancias : dict
        Diccionario con las importancias de cada variable.

    signos : dict
        Dirección esperada del efecto de cada variable:
        +1 aumenta riesgo, -1 reduce riesgo.

    baja_satisfaccion_cluster : dict
        Proporción de baja satisfacción por cluster.

    beta_0 : float
        Riesgo base del modelo.

    lambda_cluster : float
        Peso global del efecto cluster.

    eta : float
        Intensidad del efecto de satisfacción individual.

    epsilon : float
        Corrección para evitar log(0).

    Retorna
    -------
    DataFrame con probabilidad de abandono y categoría de riesgo.
    """

    df_resultado = df.copy()

    # ========================================================
    # Variables utilizadas
    # ========================================================

    variables = list(importancias.keys())

    # Verificar que las variables existan en la base
    variables_disponibles = [v for v in variables if v in df_resultado.columns]

    if len(variables_disponibles) == 0:
        raise ValueError("Ninguna de las variables de importancia está en la base.")

    # ========================================================
    # Preparar matriz X
    # ========================================================

    X = df_resultado[variables_disponibles].copy()

    # Convertir booleanos a enteros
    for col in X.select_dtypes(include=["bool"]).columns:
        X[col] = X[col].astype(int)

    # Imputar nulos con cero
    X = X.fillna(0)

    # Estandarizar variables
    scaler = StandardScaler()
    X_z = pd.DataFrame(
        scaler.fit_transform(X),
        columns=variables_disponibles,
        index=df_resultado.index
    )

    # ========================================================
    # Normalizar importancias
    # ========================================================

    suma_importancias = sum(importancias[v] for v in variables_disponibles)

    pesos = {
        v: signos[v] * (importancias[v] / suma_importancias)
        for v in variables_disponibles
    }

    # ========================================================
    # Componente de variables
    # ========================================================

    score_variables = np.zeros(len(df_resultado))

    for v in variables_disponibles:
        score_variables += pesos[v] * X_z[v].values

    # ========================================================
    # Componente de cluster
    # ========================================================

    if "cluster" not in df_resultado.columns:
        raise ValueError("La base debe contener una columna llamada 'cluster'.")

    def calcular_peso_cluster(c):
        r_c = baja_satisfaccion_cluster.get(c, np.mean(list(baja_satisfaccion_cluster.values())))

        w_c = np.log(
            (r_c + epsilon) /
            (1 - r_c + epsilon)
        )

        return w_c

    score_cluster = df_resultado["cluster"].apply(calcular_peso_cluster).values

    # ========================================================
    # Componente de satisfacción individual
    # ========================================================

    if "satisfaccion_cliente" not in df_resultado.columns:
        raise ValueError("La base debe contener una columna llamada 'satisfaccion_cliente'.")

    def calcular_gamma_satisfaccion(sat):
        if sat == "Alta satisfacción":
            return -eta
        elif sat == "Baja satisfacción":
            return eta
        elif sat == "Neutral":
            return 0
        else:
            return 0

    score_satisfaccion = df_resultado["satisfaccion_cliente"].apply(
        calcular_gamma_satisfaccion
    ).values

    # ========================================================
    # Puntaje Z
    # ========================================================

    Z = (
        beta_0
        + score_variables
        + lambda_cluster * score_cluster
        + score_satisfaccion
    )

    # ========================================================
    # Probabilidad logística
    # ========================================================

    prob_abandono = 1 / (1 + np.exp(-Z))

    df_resultado["score_variables"] = score_variables
    df_resultado["score_cluster"] = score_cluster
    df_resultado["score_satisfaccion"] = score_satisfaccion
    df_resultado["Z_abandono"] = Z
    df_resultado["prob_abandono"] = prob_abandono

    # ========================================================
    # Clasificación de riesgo
    # ========================================================

    df_resultado["nivel_riesgo_abandono"] = pd.cut(
        df_resultado["prob_abandono"],
        bins=[0, 0.25, 0.50, 0.75, 1.00],
        labels=["Bajo", "Medio", "Alto", "Crítico"],
        include_lowest=True
    )

    return df_resultado

In [ ]:
df_pred = predecir_abandono(
    df=df,
    importancias=importancias,
    signos=signos,
    baja_satisfaccion_cluster=baja_satisfaccion_cluster,
    beta_0=-1.5,
    lambda_cluster=0.7,
    eta=1.2,
    epsilon=0.001
)

In [ ]:
df_pred[
    [
        "user_id",
        "cluster",
        "satisfaccion_cliente",
        "score_variables",
        "score_cluster",
        "score_satisfaccion",
        "Z_abandono",
        "prob_abandono",
        "nivel_riesgo_abandono"
    ]
].head(20)

,user_id,cluster,satisfaccion_cliente,score_variables,score_cluster,score_satisfaccion,Z_abandono,prob_abandono,nivel_riesgo_abandono
0,USR-00001,13,Neutral,0.224247,-0.825060,0.0,-1.853295,0.135486,Bajo
1,USR-00002,10,Alta satisfacción,-0.321777,-1.434393,-1.2,-4.025852,0.017535,Bajo
2,USR-00003,13,Neutral,0.019332,-0.825060,0.0,-2.058210,0.113225,Bajo
3,USR-00004,7,Neutral,-0.570369,-0.615970,0.0,-2.501548,0.075750,Bajo
4,USR-00005,13,Neutral,0.079012,-0.825060,0.0,-1.998530,0.119357,Bajo
5,USR-00006,11,Neutral,0.182352,-0.680598,0.0,-1.794067,0.142575,Bajo
6,USR-00007,-1,Baja satisfacción,-0.347893,-0.678810,1.2,-1.123060,0.245444,Bajo
7,USR-00008,13,Neutral,-0.007681,-0.825060,0.0,-2.085223,0.110541,Bajo
8,USR-00009,12,Baja satisfacción,0.028732,-0.991789,1.2,-0.965520,0.275774,Medio
9,USR-00010,11,Baja satisfacción,0.125559,-0.680598,1.2,-0.650860,0.342796,Medio
